# Final CUDA/T4 V-14 2x core run

This recovery notebook executes only the remaining owner-authorized plan ranges `b0237` through `b0333` on a Tesla T4. It writes only the plan-bound per-coordinate batch records and their manifests to Google Drive.

It performs no aggregate, rate calculation, comparison, inference, conclusion, or final-set merge. The preceding 237 artifacts (`b0000`--`b0236`) remain required for a complete 2x set. If Colab stops, rerun the notebook with the same `RUN_LABEL`: already completed tail batches are validated and skipped. Select **T4 GPU** before running it.

In [ ]:
import shutil
import subprocess

assert shutil.which('nvidia-smi'), 'No GPU backend: Runtime -> Change runtime type -> T4 GPU, reconnect, then rerun.'
assert shutil.which('nvcc'), 'nvcc is unavailable: this is not the required CUDA/T4 runtime.'
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], text=True).strip()
assert gpu.startswith('Tesla T4,'), f'Required Tesla T4, got: {gpu}'
print('GPU:', gpu)
print(subprocess.check_output(['nvcc', '--version'], text=True))

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
import tarfile

EXPECTED_CAPSULE_SHA256 = 'ef0681f50165615cf9a862778f80ce65b58b2b643a7fe4d2612a16dc59b71337'
uploaded = files.upload()  # Select g6_cuda_t4_v14_core_final_2x_capsule.tar.gz
assert len(uploaded) == 1, 'Upload exactly one final 2x capsule.'
archive = Path(next(iter(uploaded)))
actual = hashlib.sha256(archive.read_bytes()).hexdigest()
print(f'{archive.name}: {actual}')
assert actual == EXPECTED_CAPSULE_SHA256, 'Wrong capsule; do not continue.'

with tarfile.open(archive, 'r:gz') as bundle:
    bundle.extractall('/content', filter='data')

ROOT = Path('/content/project')
CONTROL_INPUT = Path('/content/g6_v14_2x_core_control')
assert (ROOT / 'research/g6/run_cuda_v14_core_batches.py').is_file()
assert CONTROL_INPUT.is_dir()
print('Capsule extracted:', ROOT)

In [ ]:
from google.colab import drive
import json
import os
import sys

sys.dont_write_bytecode = True
drive.mount('/content/gdrive')

# This approved recovery run starts at batch index 237: b0237 through b0333.
# Its Drive folder holds only that tail; batches b0000--b0236 remain required for a complete 2x set.
RUN_LABEL = 'g6_cuda_t4_v14_core_2x_tail_from_b0237_20260921'
RUN_ROOT = Path('/content/gdrive/MyDrive') / RUN_LABEL
ARTIFACT_ROOT = RUN_ROOT / 'artifacts'
# Google Drive's FUSE mount does not implement POSIX hard links. The producer therefore
# promotes atomically on local ext4, then this notebook copies its two sealed files to Drive.
LOCAL_ARTIFACT_ROOT = Path('/content/g6_v14_core_2x_local_stage')
CONTROL_OUTPUT = RUN_ROOT / 'control'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(exist_ok=True)
LOCAL_ARTIFACT_ROOT.mkdir(exist_ok=True)
CONTROL_OUTPUT.mkdir(exist_ok=True)

EXPECTED_FILES = {
    'unbalanced_qualification.json': 'e3a95d24ea257da47c6ab8654ce137bc6ab1b35e19ac187cacd5ad2bda40b890',
    'slack_qualification.json': 'd60a90413fa7ed2ce83ce2f175ea7bddf7d9ad672ca1e7a9dbf5e93f3946278b',
    'v14_core_validation.json': 'ec959f85e665bad187f23b21ce6abea7becce2994e6889c12e9dc269a27b13f0',
    'v14_core_measurement_2x_0_4096.json': 'c55073eee05f0fcd348c4e3d57ff2b716fc1d0908971c8f27b23222b39d87173',
    'v14_core_throughput_measurement.json': '2ac1c652798ba14ea7608f09537007f748b99efc607d24e9a6b6196186f9c955',
    'v14_2x_core_batch_plan.json': '70fce6273797d5d6c3347b30010e32c8518bd09c01ce0b429d735b4df43fb215',
}
EXPECTED_PLAN_CONTENT_SHA256 = '38c01eff5a313273fc106a5f5c39760628e59132b9a7a72e0176b37aa1897748'

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for name, wanted in EXPECTED_FILES.items():
    source = CONTROL_INPUT / name
    assert source.is_file() and digest(source) == wanted, (name, digest(source) if source.exists() else 'missing', wanted)

sys.path.insert(0, str(ROOT))
from research.g6.cuda_qualification import cuda_v14_core_batch as core
from research.g6.cuda_qualification import v14_core_batch_contract as contract

plan = json.loads((CONTROL_INPUT / 'v14_2x_core_batch_plan.json').read_text(encoding='utf-8'))
assert plan['content_sha256'] == EXPECTED_PLAN_CONTENT_SHA256
by_id = contract.validate_plan(plan)
assert len(by_id) == 334
assert contract.cuda.current_hashes(contract.PRODUCER_RELATIVES) == plan['source_sha256']

def write_once(path, payload):
    if path.exists() or path.is_symlink():
        assert path.is_file() and not path.is_symlink() and path.read_bytes() == payload, f'Existing control file differs: {path}'
        return
    descriptor = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o644)
    try:
        with os.fdopen(descriptor, 'wb') as stream:
            stream.write(payload)
            stream.flush()
            os.fsync(stream.fileno())
    except BaseException:
        raise

for name in EXPECTED_FILES:
    write_once(CONTROL_OUTPUT / name, (CONTROL_INPUT / name).read_bytes())

release = {
    'schema': core.SCHEMA_RELEASE,
    'plan_content_sha256': EXPECTED_PLAN_CONTENT_SHA256,
    'released_by': 'Shivansh Srivastava',
    'date': '2026-09-21',
    'review_id': 'owner-final-cuda-t4-v14-2x-core-2026-09-21',
    'authorizes': 'run-batch',
}
release['content_sha256'] = core.content_sha256(release)
release_bytes = core.canonical_bytes(release)
write_once(CONTROL_OUTPUT / 'release_record.json', release_bytes)
assert json.loads((CONTROL_OUTPUT / 'release_record.json').read_text()) == release
core.validate_release(release, plan)

print('Plan and all control records: PASS')
print('Plan:', plan['content_sha256'])
print('Release:', release['content_sha256'])
print('Drive artifact root:', ARTIFACT_ROOT)
print('Local atomic staging root:', LOCAL_ARTIFACT_ROOT)

In [ ]:
# Final approved producer. Each batch is sealed locally on ext4, then copied to Drive.
# Completed Drive batches are validated and skipped; no final artifact is ever overwritten.
import shutil
import time
import uuid

ENV = dict(os.environ, PYTHONDONTWRITEBYTECODE='1')
unbalanced_record = CONTROL_OUTPUT / 'unbalanced_qualification.json'
slack_record = CONTROL_OUTPUT / 'slack_qualification.json'
plan_file = CONTROL_OUTPUT / 'v14_2x_core_batch_plan.json'
release_file = CONTROL_OUTPUT / 'release_record.json'
ledger = contract.base.load_v14_ledgers()['2x']['payload']
ordered_batches = sorted(plan['batches'], key=lambda batch: batch['index'])
START_BATCH_INDEX = 237
scheduled_batches = ordered_batches[START_BATCH_INDEX:]
assert len(ordered_batches) == 334 and len(scheduled_batches) == 97

def sha256_file(path):
    hasher = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            hasher.update(block)
    return hasher.hexdigest()

def transfer_sealed_batch(local, remote):
    # The producer has already validated every coordinate before sealing locally.
    # Here we verify only the byte-for-byte transfer; this avoids a redundant full re-validation.
    assert local.is_dir() and not local.is_symlink() and not remote.exists() and not remote.is_symlink()
    members = sorted(entry.name for entry in local.iterdir())
    assert members == ['manifest.json', 'records.json'], members
    local_manifest = json.loads((local / 'manifest.json').read_text(encoding='utf-8'))
    assert local_manifest['content_sha256'] == core.content_sha256(local_manifest)
    assert sha256_file(local / 'records.json') == local_manifest['records']['file_sha256']
    staging = ARTIFACT_ROOT / f'.pending-transfer-{local.name}-{uuid.uuid4().hex}'
    shutil.copytree(local, staging, copy_function=shutil.copyfile)
    try:
        assert sorted(entry.name for entry in staging.iterdir()) == members
        assert sha256_file(staging / 'records.json') == sha256_file(local / 'records.json')
        assert sha256_file(staging / 'manifest.json') == sha256_file(local / 'manifest.json')
        os.rename(staging, remote)  # Google Drive supports rename within one mounted directory.
    except BaseException:
        # Preserve the local sealed batch for retry; a remote pending directory is never treated as final.
        raise
    assert sha256_file(remote / 'records.json') == local_manifest['records']['file_sha256']
    shutil.rmtree(local)  # Safe only after Drive has the manifest-bound byte-identical copy.

def resume_sealed_batch(remote, batch):
    # A final directory appears only after a byte-verified transfer and rename in this notebook.
    # On restart, check its sealed manifest and plan/range binding without re-evaluating records already
    # validated by the producer. This keeps a resume proportional to 334 small manifests, not 1.37M records.
    assert remote.is_dir() and not remote.is_symlink()
    assert sorted(entry.name for entry in remote.iterdir()) == ['manifest.json', 'records.json']
    manifest = json.loads((remote / 'manifest.json').read_text(encoding='utf-8'))
    assert manifest['content_sha256'] == core.content_sha256(manifest)
    assert manifest['batch_id'] == batch['batch_id']
    assert manifest['plan_content_sha256'] == EXPECTED_PLAN_CONTENT_SHA256
    assert manifest['range'] == {'start': batch['start'], 'stop': batch['stop'], 'count': batch['count']}
    assert manifest['records']['record_count'] == batch['count'] and (remote / 'records.json').is_file()

completed = 0
started = time.monotonic()
for ordinal, batch in enumerate(scheduled_batches, start=START_BATCH_INDEX + 1):
    batch_id = batch['batch_id']
    remote = ARTIFACT_ROOT / batch_id
    local = LOCAL_ARTIFACT_ROOT / batch_id
    if remote.exists() or remote.is_symlink():
        resume_sealed_batch(remote, batch)
        completed += 1
        print(f'[{ordinal}/334] already complete and validated: {batch_id}', flush=True)
        continue
    if local.exists() or local.is_symlink():
        print(f'[{ordinal}/334] retrying Drive transfer of locally sealed {batch_id}', flush=True)
        transfer_sealed_batch(local, remote)
    else:
        command = [
            'python3', 'research/g6/run_cuda_v14_core_batches.py', '--run-batch',
            '--unbalanced-record', str(unbalanced_record),
            '--slack-record', str(slack_record),
            '--plan-file', str(plan_file),
            '--batch-id', batch_id,
            '--release-record', str(release_file),
            '--output-dir', str(LOCAL_ARTIFACT_ROOT),
        ]
        print(f'[{ordinal}/334] running {batch_id}: [{batch["start"]}, {batch["stop"]})', flush=True)
        result = subprocess.run(command, cwd=ROOT, env=ENV, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError(f'Batch {batch_id} failed; no later batch was started.\n\n{result.stderr[-8000:]}')
        receipt = json.loads(result.stdout)
        assert receipt['verdict'] == 'written' and receipt['batch_id'] == batch_id
        assert receipt['record_count'] == batch['count'] and Path(receipt['artifact_directory']) == local
        transfer_sealed_batch(local, remote)
    completed += 1
    elapsed_minutes = (time.monotonic() - started) / 60
    print(f'[{ordinal}/334] written and transferred: {batch_id}; tail completed={completed}/97; elapsed={elapsed_minutes:.1f} min', flush=True)

assert completed == len(scheduled_batches) == 97
print('The approved tail b0237--b0333 is present on this Drive. Stop here: this notebook intentionally does not merge, aggregate, compare, rate, or interpret artifacts.')

## If the runtime disconnects

Start a T4 runtime, reopen this notebook, and run every cell again with the same `RUN_LABEL`. The capsule and the six non-value control records are rechecked. Existing batch directories are semantically validated and skipped; only missing plan ranges run.

After all 334 ranges complete, stop. A separate owner decision is required before any whole-set validation index, aggregation, comparison, rate, or conclusion.